# L3: Agentic Sales Pipeline

<p style="background-color:#fff6e4; padding:15px; border-width:3px; border-color:#f5ecda; border-style:solid; border-radius:6px"> ⏳ <b>Note <code>(Kernel Starting)</code>:</b> This notebook takes about 30 seconds to be ready to use. You may start and watch the video while you wait.</p>

## Initial Imports

In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
import sys
sys.path.insert(0, "/Users/karli/Desktop/crew-ai-agents")

# Load environment variables
from helper import load_env
load_env()

import os
import yaml
from crewai import LLM
from crewai import Agent, Task, Crew

gemini_llm = LLM(
    model="gemini/gemini-3.5-flash",
    max_retries=6,         # Aumenta los reintentos automáticos
    timeout=120
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


<p style="background-color:#fff6ff; padding:15px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"> 💻 &nbsp; <b>Access <code>requirements.txt</code> and <code>helper.py</code> files:</b> 1) click on the <em>"File"</em> option on the top menu of the notebook and then 2) click on <em>"Open"</em>. For more help, please see the <em>"Appendix - Tips and Help"</em> Lesson.</p>

## Load API tokens for our 3rd party APIs

In [2]:
# Asegúrate de que la API key esté disponible para LiteLLM
# (LiteLLM busca automáticamente la variable GOOGLE_API_KEY en el entorno)
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")

# Configurar el LLM de Gemini usando el formato compatible con LiteLLM/CrewAI
# Reemplazamos gemini-1.5-flash por el identificador estándar actual (ej. gemini/gemini-2.5-flash)
# Configurar el LLM de Gemini con uno de los modelos disponibles en tu lista
llm_gemini = "gemini/gemini-3.5-flash"

## Loading Tasks and Agents YAML files

In [3]:
# Define file paths for YAML configurations
files = {
    'lead_agents': 'config/lead_qualification_agents.yaml',
    'lead_tasks': 'config/lead_qualification_tasks.yaml',
    'email_agents': 'config/email_engagement_agents.yaml',
    'email_tasks': 'config/email_engagement_tasks.yaml'
}

# Load configurations from YAML files
configs = {}
for config_type, file_path in files.items():
    with open(file_path, 'r') as file:
        configs[config_type] = yaml.safe_load(file)

# Assign loaded configurations to specific variables
lead_agents_config = configs['lead_agents']
lead_tasks_config = configs['lead_tasks']
email_agents_config = configs['email_agents']
email_tasks_config = configs['email_tasks']

## Create Pydantic Models for Structured Output

In [4]:
from pydantic import BaseModel, Field
from typing import Dict, Optional, List, Set, Tuple

class LeadPersonalInfo(BaseModel):
    name: str = Field(..., description="The full name of the lead.")
    job_title: str = Field(..., description="The job title of the lead.")
    role_relevance: int = Field(..., ge=0, le=10, description="A score representing how relevant the lead's role is to the decision-making process (0-10).")
    professional_background: Optional[str] = Field(..., description="A brief description of the lead's professional background.")

class CompanyInfo(BaseModel):
    company_name: str = Field(..., description="The name of the company the lead works for.")
    industry: str = Field(..., description="The industry in which the company operates.")
    company_size: int = Field(..., description="The size of the company in terms of employee count.")
    revenue: Optional[float] = Field(None, description="The annual revenue of the company, if available.")
    market_presence: int = Field(..., ge=0, le=10, description="A score representing the company's market presence (0-10).")

class LeadScore(BaseModel):
    score: int = Field(..., ge=0, le=100, description="The final score assigned to the lead (0-100).")
    scoring_criteria: List[str] = Field(..., description="The criteria used to determine the lead's score.")
    validation_notes: Optional[str] = Field(None, description="Any notes regarding the validation of the lead score.")

class LeadScoringResult(BaseModel):
    personal_info: LeadPersonalInfo = Field(..., description="Personal information about the lead.")
    company_info: CompanyInfo = Field(..., description="Information about the lead's company.")
    lead_score: LeadScore = Field(..., description="The calculated score and related information for the lead.")

## Importing Tools

In [5]:
from crewai_tools import ScrapeWebsiteTool

## Lead Qualification Crew, Agents and Tasks

In [7]:
# Creating Agents
lead_data_agent = Agent(
  config=lead_agents_config['lead_data_agent'],
  tools=[ScrapeWebsiteTool()],
  llm=gemini_llm,
    max_iter=3,              # Limita a 3 intentos por tarea
    max_execution_time=120   # Cancela la tarea si supera 2 minutos
)

cultural_fit_agent = Agent(
  config=lead_agents_config['cultural_fit_agent'],
  tools=[ScrapeWebsiteTool()]
)

scoring_validation_agent = Agent(
  config=lead_agents_config['scoring_validation_agent'],
  tools=[ScrapeWebsiteTool()]
)

# Creating Tasks
lead_data_task = Task(
  config=lead_tasks_config['lead_data_collection'],
  agent=lead_data_agent
)

cultural_fit_task = Task(
  config=lead_tasks_config['cultural_fit_analysis'],
  agent=cultural_fit_agent
)

scoring_validation_task = Task(
  config=lead_tasks_config['lead_scoring_and_validation'],
  agent=scoring_validation_agent,
  context=[lead_data_task, cultural_fit_task],
  output_pydantic=LeadScoringResult
)

# Creating Crew
lead_scoring_crew = Crew(
  agents=[
    lead_data_agent,
    cultural_fit_agent,
    scoring_validation_agent
  ],
  tasks=[
    lead_data_task,
    cultural_fit_task,
    scoring_validation_task
  ],
  verbose=True, 
  llm=gemini_llm
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## Email Engagement Crew

In [8]:
# Creating Agents
email_content_specialist = Agent(
  config=email_agents_config['email_content_specialist']
)

engagement_strategist = Agent(
  config=email_agents_config['engagement_strategist']
)

# Creating Tasks
email_drafting = Task(
  config=email_tasks_config['email_drafting'],
  agent=email_content_specialist
)

engagement_optimization = Task(
  config=email_tasks_config['engagement_optimization'],
  agent=engagement_strategist
)

# Creating Crew
email_writing_crew = Crew(
  agents=[
    email_content_specialist,
    engagement_strategist
  ],
  tasks=[
    email_drafting,
    engagement_optimization
  ],
  verbose=True
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


## Creating Complete Sales Flow

In [9]:
from crewai import Flow
from crewai.flow.flow import listen, start

class SalesPipeline(Flow):
    @start()
    def fetch_leads(self):
        # Pull our leads from the database
        leads = [
            {
                "lead_data": {
                    "name": "João Moura",
                    "job_title": "Director of Engineering",
                    "company": "Clearbit",
                    "email": "joao@clearbit.com",
                    "use_case": "Using AI Agent to do better data enrichment."
                },
            },
        ]
        return leads

    @listen(fetch_leads)
    def score_leads(self, leads):
        scores = lead_scoring_crew.kickoff_for_each(leads)
        self.state["score_crews_results"] = scores
        return scores

    @listen(score_leads)
    def store_leads_score(self, scores):
        # Here we would store the scores in the database
        return scores

    @listen(score_leads)
    def filter_leads(self, scores):
        return [score for score in scores if score['lead_score'].score > 70]

    @listen(filter_leads)
    def write_email(self, leads):
        scored_leads = [lead.to_dict() for lead in leads]
        emails = email_writing_crew.kickoff_for_each(scored_leads)
        return emails

    @listen(write_email)
    def send_email(self, emails):
        # Here we would send the emails to the leads
        return emails

flow = SalesPipeline()

## Plotting the Flow

In [10]:
flow.plot()

'/var/folders/qz/l6x_j15j41x5fn2nfl9twgp40000gn/T/crewai_flow_cy6yu4ki/crewai_flow.html'

In [11]:
from IPython.display import IFrame

IFrame(src='./crewai_flow.html', width='150%', height=600)

## Flow Kickoff

In [ ]:
import time
import os
import asyncio

# Asegurar clave de API
api_key = os.getenv("GOOGLE_API_KEY") or google_api_key
if not api_key:
    raise ValueError("GOOGLE_API_KEY is required")

os.environ["GOOGLE_API_KEY"] = api_key

# Ejecución con reintentos manuales y un timeout estricto de 120 segundos
for intento in range(3):
    try:
        print(f"Ejecutando flujo (Intento {intento + 1})...")
        
        # asyncio.wait_for cancela la ejecución si pasa de 120 segundos (2 minutos)
        emails = await asyncio.wait_for(flow.kickoff_async(), timeout=120)
        
        print("¡Ejecución completada con éxito!")
        break
    except Exception as e:
        print(f"Ocurrió un error o se agotó el tiempo límite: {e}")
        if intento < 2:
            print("Esperando 15 segundos antes de reintentar...")
            time.sleep(15)
        else:
            print("Se agotaron todos los intentos.")
            raise e

Ejecutando flujo (Intento 1)...


╭─────────────────────────────────────────────── 🌊 Flow Execution ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Starting Flow Execution                                                                                        │
│  Name: SalesPipeline                                                                                            │
│  ID: c5741d36-bcef-4e92-8de1-52aa8d011bb6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── ✨ Update Available ✨ ─────────────────────────────────────────────╮
│                                                                                                                 │
│  A new version of CrewAI is available!                                                                          │
│                                                                                                                 │
│  Current version: 1.15.33                                                                                       │
│  Latest version:  1.15.39                                                                                       │
│                                                                                                                 │
│  To update, run: uv sync --upgrade-package crewai                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🌊 Flow Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Flow Started                                                                                                   │
│  Name: SalesPipeline                                                                                            │
│  ID: c5741d36-bcef-4e92-8de1-52aa8d011bb6                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔄 Flow Method Running ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: fetch_leads                                                                                            │
│  Status: Running                                                                                                │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── ✅ Flow Method Completed ────────────────────────────────────────────╮
│                                                                                                                 │
│  Method: fetch_leads                                                                                            │
│  Status: Completed                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (76 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Cultural Fit Analyst                                                                                    │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  # Cultural Alignment & Fit Assessment Report                                                                   │
│                                                                                                                 │
│  **Lead:** João Moura, Director of Engineering                                                                  │
│  **Target Organization:** Clearbit (a HubSpot company)                                                          │
│  **Assessing Organization:** CrewAI                                                                             │
│  **Target Use Case:** Implementing AI Agents to drive next-generation B2B data enrichment                       │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 1. Executive Summary & Overall Cultural Fit Score                                                           │
│                                                                                                                 │
│  ### **Overall Cultural Fit Score: 9.6 / 10 (Exceptional Alignment)**                                           │
│                                                                                                                 │
│  The partnership between **Clearbit** and **CrewAI** represents an exceptionally high cultural and strategic    │
│  synergy. Clearbit has built its brand on engineering excellence, an API-first developer ethos, data            │
│  precision, and progressive automation. CrewAI is pioneered on multi-agent collaboration, open developer        │
│  empowerment, and state-of-the-art agentic orchestration.                                                       │
│                                                                                                                 │
│  Both organizations share a common worldview: complex, knowledge-intensive manual workflows should be           │
│  abstracted into autonomous, intelligent, and highly scalable software systems. Led at the technical helm by    │
│  João Moura (Director of Engineering), Clearbit's engineering organization possesses the cultural agility,      │
│  technical sophistication, and innovation-first mindset required to adopt, co-innovate, and scale CrewAI's      │
│  multi-agent platform seamlessly.                                                                               │
│                                                                                                                 │
│  ---                                                                                                            │
│                                                                                                                 │
│  ## 2. Cultural Values Analysis                                                                                 │
│                                                                                                                 │
│  ### A. Innovation & Technological Craftsmanship                                                                │
│  * **Clearbit:** Clearbit’s engineering reputation is g

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Assess the cultural alignment between the lead's company and our organization by considering the         │
│  following:                                                                                                     │
│    - Cultural Values: Analyze the company's publicly stated values and internal culture (e.g., innovation,      │
│  sustainability, employee engagement).                                                                          │
│    - Strategic Alignment: Evaluate how well the company's goals and mission align with our organization's       │
│  strategic objectives.                                                                                          │
│    - Qualitative Scoring: Assign a qualitative score (0-10) representing the overall cultural fit.              │
│    - Comments: Provide additional comments or observations that support the cultural fit score.                 │
│                                                                                                                 │
│  - Our Company and Product:                                                                                     │
│    - Company Name: CrewAI                                                                                       │
│    - Product: Multi-Agent Orchestration Platform                                                                │
│    - ICP: Enterprise companies looking into Agentic automation.                                                 │
│    - Pitch: We are a platform that allows you to orchestrate AI Agents for automations to any vertical.         │
│                                                                                                                 │
│  - Lead Data:                                                                                                   │
│    {'name': 'João Moura', 'job_title': 'Director of Engineering', 'company': 'Clearbit', 'email':               │
│  'joao@clearbit.com', 'use_case': 'Using AI Agent to do better data enrichment.'}                               │
│                                                                                                                 │
│  Agent: Cultural Fit Analyst                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Aggregate the collected data and perform the following steps: - Score Calculation: Based on predefined   │
│  criteria, calculate a final lead score (0-100). Consider factors such as:                                      │
│    - Role Relevance                                                                                             │
│    - Company Size                                                                                               │
│    - Market Presence                                                                                            │
│    - Cultural Fit                                                                                               │
│  - Scoring Criteria Documentation: List the criteria used to determine the score. - Validation: Review the      │
│  collected data and the calculated score for consistency and accuracy. Make adjustments if necessary. - Final   │
│  Report: Compile a summary report that includes the final validated lead score, the criteria used, and any      │
│  validation notes.                                                                                              │
│  - Our Company and Product:                                                                                     │
│    - Company Name: CrewAI                                                                                       │
│    - Product: Multi-Agent Orchestration Platform                                                                │
│    - ICP: Enterprise companies looking into Agentic automation.                                                 │
│    - Pitch: We are a platform that allows you to orchestrate AI Agents for automations to any vertical.         │
│                                                                                                                 │
│  - Lead Data:                                                                                                   │
│    {'name': 'João Moura', 'job_title': 'Director of Engineering', 'company': 'Clearbit', 'email':               │
│  'joao@clearbit.com', 'use_case': 'Using AI Agent to do better data enrichment.'}                               │
│                                                                                                                 │
│  ID: dfa63194-c13b-4cdb-81b8-5ff2ef8c3e0d                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Lead Scorer and Validator                                                                               │
│                                                                                                                 │
│  Task: Aggregate the collected data and perform the following steps: - Score Calculation: Based on predefined   │
│  criteria, calculate a final lead score (0-100). Consider factors such as:                                      │
│    - Role Relevance                                                                                             │
│    - Company Size                                                                                               │
│    - Market Presence                                                                                            │
│    - Cultural Fit                                                                                               │
│  - Scoring Criteria Documentation: List the criteria used to determine the score. - Validation: Review the      │
│  collected data and the calculated score for consistency and accuracy. Make adjustments if necessary. - Final   │
│  Report: Compile a summary report that includes the final validated lead score, the criteria used, and any      │
│  validation notes.                                                                                              │
│  - Our Company and Product:                                                                                     │
│    - Company Name: CrewAI                                                                                       │
│    - Product: Multi-Agent Orchestration Platform                                                                │
│    - ICP: Enterprise companies looking into Agentic automation.                                                 │
│    - Pitch: We are a platform that allows you to orchestrate AI Agents for automations to any vertical.         │
│                                                                                                                 │
│  - Lead Data:                                                                                                   │
│    {'name': 'João Moura', 'job_title': 'Director of Engineering', 'company': 'Clearbit', 'email':               │
│  'joao@clearbit.com', 'use_case': 'Using AI Agent to do better data enrichment.'}                               │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Usage Metrics and Costs

Let’s see how much it would cost each time if this crew runs at scale.

In [14]:
""" import pandas as pd

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([flow.state["score_crews_results"][0].token_usage.dict()])

# Calculate total costs
costs = 0.150 * df_usage_metrics['total_tokens'].sum() / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Display the DataFrame
df_usage_metrics """

# 1. Simula la respuesta que entregaría el kickoff()
mock_crew_output = """
{
    "lead_name": "João Moura",
    "job_title": "Director of Engineering",
    "company": "Clearbit",
    "role_relevance_score": 9,
    "company_size_score": 8,
    "cultural_fit_score": 9,
    "final_lead_score": 8.7,
    "status": "Qualified"
}
"""

# 2. Usa este mock directamente en tu función de métricas/scoring
def calcular_metricas_pipeline(crew_result):
    print("Procesando métricas sobre el resultado...")
    # Aquí desarrollas toda tu lógica de métricas, dataframes o reportes
    return {"procesado": True, "data": crew_result}

# Pruebas tu código inmediatamente sin esperas
resultado_metricas = calcular_metricas_pipeline(mock_crew_output)
print(resultado_metricas)

Procesando métricas sobre el resultado...
{'procesado': True, 'data': '\n{\n    "lead_name": "João Moura",\n    "job_title": "Director of Engineering",\n    "company": "Clearbit",\n    "role_relevance_score": 9,\n    "company_size_score": 8,\n    "cultural_fit_score": 9,\n    "final_lead_score": 8.7,\n    "status": "Qualified"\n}\n'}


In [15]:
import pandas as pd

# Convert UsageMetrics instance to a DataFrame
df_usage_metrics = pd.DataFrame([emails[0].token_usage.dict()])

# Calculate total costs
costs = 0.150 * df_usage_metrics['total_tokens'].sum() / 1_000_000
print(f"Total costs: ${costs:.4f}")

# Display the DataFrame
df_usage_metrics

NameError: name 'emails' is not defined

## Inspecting Results

In [16]:
scores = flow.state["score_crews_results"]

KeyError: 'score_crews_results'

In [ ]:
import pandas as pd
from IPython.display import display, HTML

lead_scoring_result = scores[0].pydantic

# Create a dictionary with the nested structure flattened
data = {
    'Name': lead_scoring_result.personal_info.name,
    'Job Title': lead_scoring_result.personal_info.job_title,
    'Role Relevance': lead_scoring_result.personal_info.role_relevance,
    'Professional Background': lead_scoring_result.personal_info.professional_background,
    'Company Name': lead_scoring_result.company_info.company_name,
    'Industry': lead_scoring_result.company_info.industry,
    'Company Size': lead_scoring_result.company_info.company_size,
    'Revenue': lead_scoring_result.company_info.revenue,
    'Market Presence': lead_scoring_result.company_info.market_presence,
    'Lead Score': lead_scoring_result.lead_score.score,
    'Scoring Criteria': ', '.join(lead_scoring_result.lead_score.scoring_criteria),
    'Validation Notes': lead_scoring_result.lead_score.validation_notes
}

# Convert the dictionary to a DataFrame
df = pd.DataFrame.from_dict(data, orient='index', columns=['Value'])

# Reset the index to turn the original column names into a regular column
df = df.reset_index()

# Rename the index column to 'Attribute'
df = df.rename(columns={'index': 'Attribute'})

# Create HTML table with bold attributes and left-aligned values
html_table = df.style.set_properties(**{'text-align': 'left'}) \
                     .format({'Attribute': lambda x: f'<b>{x}</b>'}) \
                     .hide(axis='index') \
                     .to_html()

# Display the styled HTML table
display(HTML(html_table))

## Results

In [ ]:
import textwrap

result_text = emails[0].raw
wrapped_text = textwrap.fill(result_text, width=80)
print(wrapped_text)

## How Complex Can it Get?

In [17]:
from crewai import Flow
from crewai.flow.flow import listen, start, and_, or_, router

class SalesPipeline(Flow):
    
  @start()
  def fetch_leads(self):
    # Pull our leads from the database
    # This is a mock, in a real-world scenario, this is where you would
    # fetch leads from a database
    leads = [
      {
        "lead_data": {
          "name": "João Moura",
          "job_title": "Director of Engineering",
          "company": "Clearbit",
          "email": "joao@clearbit.com",
          "use_case": "Using AI Agent to do better data enrichment."
        },
      },
    ]
    return leads

  @listen(fetch_leads)
  def score_leads(self, leads):
    scores = lead_scoring_crew.kickoff_for_each(leads)
    self.state["score_crews_results"] = scores
    return scores

  @listen(score_leads)
  def store_leads_score(self, scores):
    # Here we would store the scores in the database
    return scores

  @listen(score_leads)
  def filter_leads(self, scores):
    return [score for score in scores if score['lead_score'].score > 70]

  @listen(and_(filter_leads, store_leads_score))
  def log_leads(self, leads):
    print(f"Leads: {leads}")

  @router(filter_leads)
  def count_leads(self, scores):
    if len(scores) > 10:
      return 'high'
    elif len(scores) > 5:
      return 'medium'
    else:
      return 'low'

  @listen('high')
  def store_in_salesforce(self, leads):
    return leads

  @listen('medium')
  def send_to_sales_team(self, leads):
    return leads

  @listen('low')
  def write_email(self, leads):
    scored_leads = [lead.to_dict() for lead in leads]
    emails = email_writing_crew.kickoff_for_each(scored_leads)
    return emails

  @listen(write_email)
  def send_email(self, emails):
    # Here we would send the emails to the leads
    return emails

TypeError: router() got an unexpected keyword argument 'paths'

## Plotting the Flow

In [ ]:
flow = SalesPipeline()
flow.plot()

In [ ]:
from IPython.display import IFrame

IFrame(src='./crewai_flow_complex.html', width='150%', height=600)